# TITLE PLACE HOLDER

SOME TEXT HERE

## Pre processing of the datasets

SOME TEXT HERE

In [ ]:
import pandas as pd;
import numpy as np;

data = pd.read_excel("Data/meta Niloofar 8.10.404.final.xlsx");
data = data.drop(columns=['Column1', 'N0.']);

# Forward fill these categorical features
data["Author"] = data["Author"].ffill();
data["Cancer"] = data["Cancer"].ffill();
data["Cell Line"] = data["Cell Line"].ffill();

# 1. Remove all whitespace
data["Unit"] = data["Unit"].str.replace(r"\s+", "", regex=True);
data["Time"] = data["Time"].str.replace(r"\s+", "", regex=True);

# 2. Normalize unicode µ signs to standard 'µ'
data["Unit"] = data["Unit"].str.replace('μ', 'µ')

# 3. Lowercase for consistency
data["Unit"] = data["Unit"].str.lower();
data["Time"] = data["Time"].str.lower();


print(data["Unit"].unique())
print(data["Unit"].value_counts())

# Molecular weight of Osthole.
MW = 244.28;

# The new data (Osthole data) needs unit transformation.
def to_uM(row):
    if row["Unit"].strip() == "mg/ml":
        return row["Dose"] * 1e6 / MW;

    elif row["Unit"].strip() == "nm":
        return row["Dose"] * 1e-3;

    elif row["Unit"].strip() == "µg/ml":
        return row['Dose'] * 1e3 / MW;

    else:
        return row["Dose"];

data["Dose"] = data.apply(to_uM, axis=1)
data["Dose"] = data["Dose"].round(2);

data["Unit"] = "µM"
data["Coumarin"] = "Osthole"
col = data.pop("Coumarin");
data.insert(3, 'Coumarin', col);
col = data.pop("Time");
data.insert(5, 'Time', col);

data.to_csv("Data/cleanedData.csv", index=False);

# Now we shall concatenate this data to the old one.

# Load the old dataset

oldData = pd.read_csv("Data/oldData.csv");

# print(oldData.columns);

# Just some re ordering and renaming of the feaatures to concat to the new osthole data easily
oldData = oldData[['Seo', 'Cancer Type', 'Cell Line', 'Coumarin Type', 'Coumarin Dose', 'Time', 'Viability']];
oldData.columns = ['Author', 'Cancer', 'Cell Line', 'Coumarin', 'Dose', 'Time', "Viability"];
newData = data[['Author', 'Cancer', 'Cell Line', 'Coumarin', 'Dose', 'Time', "Viability"]]

# Make the time feature numerical
newData['Time'] = newData['Time'].str.replace(r'\D', '', regex=True).astype(int);

# Concat the datas toghether
finalDataset = pd.concat([oldData, newData]);

finalDataset['Cancer'] = finalDataset['Cancer'].str.strip().str.lower().str.replace('\n', ' ', regex=False);

mapping = {
    # Prostate
    'prostate': 'prostate',
    'prostate cancer': 'prostate',

    # Breast
    'breast': 'breast',

    # Lung
    'lung': 'lung',
    'lung cancer': 'lung',
    'lung adenocarcinoma': 'lung',

    # Colon / Colorectal
    'colon': 'colon',
    'colorectal cancer': 'colon',
    'colon adenocarcinoma': 'colon',

    # Liver
    'liver': 'liver',
    'hepatocellular carcinoma': 'liver',

    # Cholangiocarcinoma
    'cholangiocarcinoma': 'cholangiocarcinoma',
    'intrahepatic cholangiocarcinoma': 'cholangiocarcinoma',

    # Cervical
    'cervical': 'cervical',
    'cervix': 'cervical',
    'cervical carcinoma': 'cervical',

    # Ovarian
    'ovarian': 'ovarian',
    'ovarian carcinoma': 'ovarian',
    'ovarian endometrioid adenocarcinoma': 'ovarian',

    # Endometrial / Uterine
    'uterine': 'endometrial',
    'endometrial': 'endometrial',

    # Renal
    'renal': 'renal',
    'renal cell carcinoma': 'renal',

    # Bone
    'bone': 'bone',
    'osteosarcoma': 'bone',

    # Skin
    'skin': 'skin',
    'melanoma': 'melanoma',

    # Brain tumors
    'glioma': 'glioma',
    'glioblastoma': 'glioma',
    'oligodendroglioma': 'glioma',
    'medulloblastoma': 'medulloblastoma',

    # Other organs
    'gastric': 'gastric',
    'pancreatic': 'pancreatic',
    'oral': 'oral',
    'salivary gland': 'salivary gland',
    'bladder': 'bladder',
    'nasopharyngeal': 'nasopharyngeal',
    'esophageal squamous cell carcinoma': 'esophageal',
    'gallbladder cancer': 'gallbladder',
    'leukemia/lymphoma': 'leukemia:lymphoma'
}

finalDataset['Cancer'] = finalDataset['Cancer'].replace(mapping);
finalDataset.columns = finalDataset.columns.str.replace(" ", "", regex=False);
finalDataset.columns = finalDataset.columns.str.strip();


print(finalDataset['Cancer'].unique())
print(finalDataset['Cancer'].value_counts())

finalDataset.to_csv("Data/Final_Data.csv", index = False);

print(finalDataset["Coumarin"].value_counts())

## Feature Engineering

The descriptor features of the coumarins in the dataset will be defined here and will be added to
the dataset as new features.

In [2]:
import numpy as np
import pandas as pd

# All features Engineering will be done in this script, and kept separated from the general preProcessing

# The first obvious feature to add will be the Molecular Weights (From PubChem)
MWs = {"Auraptene": 298.4,
       "Osthole": 244.28,
       "Esculetin": 178.14,
       "GalbanicAcid": 398.5,
       "Umbelliprenin": 366.5};

# The second Important Feature will be LogP (XLogP3 From PubChem)
LogPs = {"Auraptene": 5.3,
         "Osthole": 3.8,
         "Esculetin": 1.2,
         "GalbanicAcid": 5.1,
         "Umbelliprenin": 7.1}

# Number of the Hydrogen bond donors
numHDonors = {"Auraptene": 0,
              "Osthole": 0,
              "Esculetin": 2,
              "GalbanicAcid": 1,
              "Umbelliprenin": 0}

# Number of the Hydrohen bond acceptors
numHAcceptors = {"Auraptene": 3,
                 "Osthole": 3,
                 "Esculetin": 4,
                 "GalbanicAcid": 5,
                 "Umbelliprenin": 3}

# Number of rotatable bonds
rotatableBonds = {"Auraptene": 6,
                  "Osthole": 3,
                  "Esculetin": 0,
                  "GalbanicAcid": 6,
                  "Umbelliprenin": 9}

# Topological Polar Surface Area (From PubChem)
TPSAs = {"Auraptene": 35.5,
         "Osthole": 35.5,
         "Esculetin": 66.8,
         "GalbanicAcid": 72.8,
         "Umbelliprenin": 35.5}

# Ring Counts
ringCounts = {"Auraptene": 2,
              "Osthole": 2,
              "Esculetin": 2,
              "GalbanicAcid": 3,
              "Umbelliprenin": 2}

# FractionCsp3: The fraction of carbon atoms that are sp3 hybridized.
fCsp3s = {"Auraptene": 0.32,
          "Osthole": 0.27,
          "Esculetin": 0,
          "GalbanicAcid": 0.5,
          "Umbelliprenin": 0.38}


# Read the data
data = pd.read_csv("Data/Final_Data.csv")

# Add the new features
data["MW"] = data["Coumarin"].map(MWs)
data["LogP"] = data["Coumarin"].map(LogPs)
data["NumHDonors"] = data["Coumarin"].map(numHDonors)
data["NumHAcceptors"] = data["Coumarin"].map(numHAcceptors)
data["RotatableBonds"] = data["Coumarin"].map(rotatableBonds)
data["TPSA"] = data["Coumarin"].map(TPSAs)
data["RingCounts"] = data["Coumarin"].map(ringCounts)
data["fCsp3"] = data["Coumarin"].map(fCsp3s)

print(data.head)

<bound method NDFrame.head of                   Author       Cancer CellLine   Coumarin   Dose  Time  \
0     Abolhassani et al.     prostate     PC-3  Auraptene    5.0    48   
1     Abolhassani et al.     prostate     PC-3  Auraptene   10.0    48   
2     Abolhassani et al.     prostate     PC-3  Auraptene   20.0    48   
3     Abolhassani et al.     prostate     PC-3  Auraptene   40.0    48   
4     Abolhassani et al.     prostate     PC-3  Auraptene   80.0    48   
...                  ...          ...      ...        ...    ...   ...   
1692    Tian Le Zou 2019  gallbladder  SGC-996    Osthole   25.0    72   
1693    Tian Le Zou 2019  gallbladder  SGC-996    Osthole   50.0    72   
1694    Tian Le Zou 2019  gallbladder  SGC-996    Osthole  100.0    72   
1695    Tian Le Zou 2019  gallbladder  SGC-996    Osthole  150.0    72   
1696    Tian Le Zou 2019  gallbladder  SGC-996    Osthole  200.0    72   

      Viability      MW  LogP  NumHDonors  NumHAcceptors  RotatableBonds  \
0    